In [58]:
import os
import re
import logging
import json
from params.paths import ROOT_DIR
from datetime import datetime, timedelta
from typing import Iterator, Tuple, List
import psycopg2
from dotenv import load_dotenv
import time
from tqdm import tqdm

from file_handling.file_read_writer import read_json, write_json

REPR_SPEECHES_DIR = os.path.join(ROOT_DIR, 'data', 'data_repr_speeches')
TARGET_DIR = os.path.join(ROOT_DIR, 'data', 'data_speeches')
print(os.listdir(REPR_SPEECHES_DIR))

['無所属', '自民', '国民', 'summary.json', '民主', 'Ｎ党', '公明', '立憲', 'れ新', 'diachronic_stats.json', 'LGBT_summaries.hdf5', '参政', '保守', '維新', '無', '共産', '沖縄', '有志']


In [59]:
def iterate_speech_files(root_dir: str) -> Iterator[Tuple[str, str, str,str]]:
	for party in os.listdir(root_dir):
		party_dir = os.path.join(root_dir, party)
		if not os.path.isdir(party_dir):
			continue
		for repr in os.listdir(party_dir):
			repr_dir = os.path.join(party_dir, repr)
			if not os.path.isdir(repr_dir):
				continue
			for topic_dir in os.listdir(repr_dir):
				topic_json_path = os.path.join(repr_dir, topic_dir, 'opinions.json')
				if not os.path.exists(topic_json_path):
					continue
				opinions_json = read_json(topic_json_path)
				if not opinions_json:
					continue
				yield topic_json_path, repr, party, topic_dir


def get_repr_data_from_db(repr: str, cur: psycopg2.extensions.cursor) -> List[Tuple[str, str, str]]:
	cur.execute("SELECT * FROM person WHERE name_kanji LIKE %s", (repr,))
	return cur.fetchall()


In [ ]:
load_dotenv()

conn = None
try:
    conn = psycopg2.connect(
        dbname="kokkaidoc",
        user="postgres",
        password=os.getenv("PSQL_DATABASE_PASSWORD"),
        host="localhost",
        port="5432",
    )
    print("Connected.")
    non_existing_reprs = []

    with conn.cursor() as cur:
        for speech_file_path, repr, party, topic_name in iterate_speech_files(REPR_SPEECHES_DIR):
            print("Working on", repr, party, topic_name)
            repr_data = get_repr_data_from_db(repr, cur)
            if not repr_data:
                os.rmdir(speech_file_path)
                non_existing_reprs.append({'repr': repr, 'party': party, 'topic': topic_name})
                print("Representative not found in DB, skipping")
                continue
            repr_id = repr_data[0][0]
            repr_dir = os.path.join(TARGET_DIR, f"{repr}_{repr_id}")
            os.makedirs(repr_dir, exist_ok=True)
            topic_target_dir = os.path.join(repr_dir, topic_name)
            os.makedirs(topic_target_dir, exist_ok=True)
            print("Outputting to", topic_target_dir)
            speech_file_data = read_json(speech_file_path)
            search_words = speech_file_data['search_words']
            speeches = speech_file_data['speeches']
            if os.path.exists(os.path.join(topic_target_dir, 'opinions.jsonl')):
                print("Speeches already exist, skipping")
                continue
            print("Writing speeches")
            with open(os.path.join(topic_target_dir, 'opinions.jsonl'), 'w', encoding='utf-8') as f:
                for speech in speeches:
                    json.dump(speech, f, ensure_ascii=False)
                    f.write('\n')
            print("Writing search words")
            with open(os.path.join(topic_target_dir, 'search_words.json'), 'w', encoding='utf-8') as f:
                json.dump({'search_words': search_words}, f, ensure_ascii=False)



    

    conn.commit()
    print("These representatives do not exist in DB, recommending to create entries for them:")
    print("".join(f"{r['repr']} {r['party']} {r['topic']}\n" for r in non_existing_reprs))

except Exception as e:
    if conn:
        conn.rollback()
    raise
finally:
    if conn:
        conn.close()

Connected.
Working on 大野泰正 無所属 少子化
Outputting to /root/projects/kokkai_analysis/data/params/../data/data_speeches/大野泰正_3874/少子化
Speeches already exist, skipping
Working on 大野泰正 無所属 物価高対策・減税と賃上げ
Outputting to /root/projects/kokkai_analysis/data/params/../data/data_speeches/大野泰正_3874/物価高対策・減税と賃上げ
Speeches already exist, skipping
Working on 大野泰正 無所属 マイナンバー
Outputting to /root/projects/kokkai_analysis/data/params/../data/data_speeches/大野泰正_3874/マイナンバー
Speeches already exist, skipping
Working on 大野泰正 無所属 気候変動
Outputting to /root/projects/kokkai_analysis/data/params/../data/data_speeches/大野泰正_3874/気候変動
Speeches already exist, skipping
Working on 大野泰正 無所属 原発
Outputting to /root/projects/kokkai_analysis/data/params/../data/data_speeches/大野泰正_3874/原発
Speeches already exist, skipping
Working on 大野泰正 無所属 防衛
Outputting to /root/projects/kokkai_analysis/data/params/../data/data_speeches/大野泰正_3874/防衛
Speeches already exist, skipping
Working on 長浜博行 無所属 少子化
Outputting to /root/projects/kokkai_analysi